# 03 — H&M Popularity Baseline — Fixed Full Version

This full notebook includes MRR@K, per-K coverage, and garment-group popularity baseline support.


In [1]:
import os
import json
import platform
import math
import time
from pathlib import Path
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


Python: 3.12.12
Pandas: 2.3.3
NumPy: 2.0.2


## 1. Environment and paths


In [2]:
def detect_environment() -> str:
    """
    Robust environment detection.
    Kaggle is checked first because some Kaggle notebook environments can import google.colab,
    but Google Drive mounting is not supported there.
    """
    if (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
        or Path("/kaggle/input").exists()
        or Path("/kaggle/working").exists()
    ):
        return "kaggle"

    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"

    return "local"


ENVIRONMENT = detect_environment()
IN_COLAB = ENVIRONMENT == "colab"
IN_KAGGLE = ENVIRONMENT == "kaggle"

print("Detected environment:", ENVIRONMENT)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Google Drive mount because this is not real Colab.")


Detected environment: kaggle
Skipping Google Drive mount because this is not real Colab.


In [3]:
PROJECT_NAME = "hm-recommender"


def find_processed_data_dir() -> Path:
    """
    Finds the folder containing the processed H&M parquet files.
    Works in Kaggle, Colab, and VSCode/local.
    """
    expected_file = "train_interactions.parquet"

    candidate_dirs = [
        Path("/kaggle/working") / PROJECT_NAME / "data" / "processed",
        Path("/content/drive/MyDrive") / PROJECT_NAME / "data" / "processed",
        Path.cwd() / "data" / "processed",
        Path.cwd() / PROJECT_NAME / "data" / "processed",
    ]

    for candidate in candidate_dirs:
        if (candidate / expected_file).exists():
            return candidate

    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob(expected_file))
        if matches:
            return matches[0].parent

    raise FileNotFoundError(
        "Could not find processed data folder. "
        "Run 01_hm_preprocessing.ipynb first or attach its output dataset."
    )


PROCESSED_DATA_DIR = find_processed_data_dir()

if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd()

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
BASELINE_ARTIFACTS_DIR = ARTIFACTS_DIR / "popularity_baseline"
BASELINE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("BASELINE_ARTIFACTS_DIR:", BASELINE_ARTIFACTS_DIR)


PROJECT_DIR: /kaggle/working/hm-recommender
PROCESSED_DATA_DIR: /kaggle/input/datasets/albaky7/hm-recommender/hm-recommender/data/processed
BASELINE_ARTIFACTS_DIR: /kaggle/working/hm-recommender/artifacts/popularity_baseline


In [4]:
required_processed_files = [
    "train_interactions.parquet",
    "val_interactions.parquet",
    "test_interactions.parquet",
    "articles_processed.parquet",
    "article_id_map.parquet",
    "sample_submission_processed.parquet",
]

missing = []

for filename in required_processed_files:
    path = PROCESSED_DATA_DIR / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"Found {filename}: {size_mb:.2f} MB")
    else:
        missing.append(str(path))

if missing:
    missing_text = "\n".join(missing)
    raise FileNotFoundError(
        "These processed files are missing. Run 01_hm_preprocessing.ipynb first.\n"
        + missing_text
    )

print("All required processed files found.")


Found train_interactions.parquet: 134.36 MB
Found val_interactions.parquet: 28.11 MB
Found test_interactions.parquet: 28.72 MB
Found articles_processed.parquet: 6.66 MB
Found article_id_map.parquet: 1.20 MB
Found sample_submission_processed.parquet: 84.86 MB
All required processed files found.


## 2. Configuration


In [5]:
TOP_K_VALUES = [12, 20]
PRIMARY_K = 20

# More candidates are scanned so that already-purchased items can be filtered out.
N_POPULAR_CANDIDATES = 5000

# Demo output size. This does not affect model evaluation.
N_DEMO_USERS = 1000

# Leave this False unless you want a full Kaggle-style CSV for all sample submission customers.
GENERATE_FULL_SAMPLE_SUBMISSION = False

RECENT_WINDOWS_DAYS = [30, 90]
HALF_LIFE_DAYS = 30.0
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("TOP_K_VALUES:", TOP_K_VALUES)
print("PRIMARY_K:", PRIMARY_K)
print("N_POPULAR_CANDIDATES:", N_POPULAR_CANDIDATES)
print("RECENT_WINDOWS_DAYS:", RECENT_WINDOWS_DAYS)
print("HALF_LIFE_DAYS:", HALF_LIFE_DAYS)


TOP_K_VALUES: [12, 20]
PRIMARY_K: 20
N_POPULAR_CANDIDATES: 5000
RECENT_WINDOWS_DAYS: [30, 90]
HALF_LIFE_DAYS: 30.0


## 3. Utility functions


In [6]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, show_head: bool = True) -> None:
    print()
    print(name)
    print("-" * 90)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    if show_head:
        display(df.head())


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved: {path}")


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"Saved: {path} | {size_mb:.2f} MB")


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"Saved: {path} | {size_mb:.2f} MB")


In [7]:
def apk(recommended: List[int], actual: set, k: int) -> float:
    """Average Precision at K for one user."""
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen_recommended = set()

    for rank, item in enumerate(recommended[:k], start=1):
        if item in seen_recommended:
            continue
        seen_recommended.add(item)

        if item in actual:
            hits += 1
            score += hits / rank

    return score / min(len(actual), k)


def ndcgk(recommended: List[int], actual: set, k: int) -> float:
    """NDCG@K for one user with binary relevance."""
    if not actual:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(actual), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0


def precisionk(recommended: List[int], actual: set, k: int) -> float:
    if k <= 0:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / k


def recallk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / len(actual)


def mrrk(recommended: List[int], actual: set, k: int) -> float:
    """Mean Reciprocal Rank at K for one user."""
    if not actual:
        return 0.0

    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            return 1.0 / rank

    return 0.0


def hitratek(recommended: List[int], actual: set, k: int) -> float:
    return float(any(item in actual for item in recommended[:k]))


In [8]:
def build_user_item_sets(df: pd.DataFrame) -> Dict[int, set]:
    """Build {customer_idx: set(article_idx)} from an interaction dataframe."""
    grouped = df.groupby("customer_idx", sort=False)["article_idx"].agg(
        lambda values: set(values.to_numpy(dtype=np.int32))
    )
    return grouped.to_dict()


def recommend_from_ranked_items(
    ranked_items: np.ndarray,
    seen_items: Optional[set],
    k: int,
    n_candidates: int = 5000,
) -> List[int]:
    """
    Generate top-K recommendations from a fixed item ranking.
    If seen_items is provided, already-purchased training items are filtered out.
    """
    if seen_items is None or len(seen_items) == 0:
        return ranked_items[:k].astype(int).tolist()

    recommendations = []
    scan_limit = min(len(ranked_items), n_candidates)

    for item in ranked_items[:scan_limit]:
        item_int = int(item)
        if item_int not in seen_items:
            recommendations.append(item_int)
            if len(recommendations) == k:
                break

    if len(recommendations) < k:
        for item in ranked_items[scan_limit:]:
            item_int = int(item)
            if item_int not in seen_items:
                recommendations.append(item_int)
                if len(recommendations) == k:
                    break

    return recommendations


In [9]:
def evaluate_fixed_ranking(
    ranked_items: np.ndarray,
    train_seen_by_user: Dict[int, set],
    eval_actual_by_user: Dict[int, set],
    k_values: List[int],
    model_name: str,
    eval_name: str,
    warm_start_only: bool = False,
    total_catalog_items: Optional[int] = None,
) -> Tuple[pd.DataFrame, Dict[int, List[int]]]:
    """
    Evaluate a fixed popularity ranking without creating a giant prediction table.
    Coverage is calculated separately for each K.
    """
    started = time.time()

    if warm_start_only:
        user_ids = [u for u in eval_actual_by_user.keys() if u in train_seen_by_user]
    else:
        user_ids = list(eval_actual_by_user.keys())

    if not user_ids:
        raise ValueError(f"No users found for {model_name}, {eval_name}, warm_start={warm_start_only}")

    max_k = max(k_values)

    sums = {
        k: {
            "precision": 0.0,
            "recall": 0.0,
            "map": 0.0,
            "ndcg": 0.0,
            "mrr": 0.0,
            "hitrate": 0.0,
        }
        for k in k_values
    }

    unique_recommended_items_by_k = {k: set() for k in k_values}
    sample_recommendations_by_user = {}

    for idx, user_id in enumerate(user_ids):
        actual = eval_actual_by_user[user_id]
        seen = train_seen_by_user.get(user_id)

        recommended = recommend_from_ranked_items(
            ranked_items=ranked_items,
            seen_items=seen,
            k=max_k,
            n_candidates=N_POPULAR_CANDIDATES,
        )

        if idx < 10:
            sample_recommendations_by_user[int(user_id)] = recommended

        for k in k_values:
            top_k_recommended = recommended[:k]
            unique_recommended_items_by_k[k].update(top_k_recommended)

            sums[k]["precision"] += precisionk(recommended, actual, k)
            sums[k]["recall"] += recallk(recommended, actual, k)
            sums[k]["map"] += apk(recommended, actual, k)
            sums[k]["ndcg"] += ndcgk(recommended, actual, k)
            sums[k]["mrr"] += mrrk(recommended, actual, k)
            sums[k]["hitrate"] += hitratek(recommended, actual, k)

    rows = []
    n_users = len(user_ids)
    elapsed_seconds = time.time() - started

    for k in k_values:
        row = {
            "model_name": model_name,
            "eval_split": eval_name,
            "warm_start_only": warm_start_only,
            "k": k,
            "n_eval_users": n_users,
            "precision_at_k": sums[k]["precision"] / n_users,
            "recall_at_k": sums[k]["recall"] / n_users,
            "map_at_k": sums[k]["map"] / n_users,
            "ndcg_at_k": sums[k]["ndcg"] / n_users,
            "mrr_at_k": sums[k]["mrr"] / n_users,
            "hitrate_at_k": sums[k]["hitrate"] / n_users,
            "unique_recommended_items": len(unique_recommended_items_by_k[k]),
            "elapsed_seconds": round(elapsed_seconds, 2),
        }

        if total_catalog_items is not None and total_catalog_items > 0:
            row["catalog_coverage"] = len(unique_recommended_items_by_k[k]) / total_catalog_items
        else:
            row["catalog_coverage"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows), sample_recommendations_by_user


## 4. Load processed data


In [10]:
train_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

val_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "val_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

test_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "test_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
sample_submission = pd.read_parquet(PROCESSED_DATA_DIR / "sample_submission_processed.parquet")

print_df_info(train_interactions, "Train interactions")
print_df_info(val_interactions, "Validation interactions")
print_df_info(test_interactions, "Test interactions")
print_df_info(article_id_map, "Article ID map")
print_df_info(articles, "Articles processed")
print_df_info(sample_submission, "Sample submission processed")



Train interactions
------------------------------------------------------------------------------------------
Shape: (22235277, 6)
Memory usage: 530.13 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2018-09-20,2,40179,0.050831,2,1.0
1,2018-09-20,7,46302,0.016932,2,1.0
2,2018-09-20,7,6386,0.020322,2,1.0
3,2018-09-20,198,47416,0.030492,1,1.0
4,2018-09-20,198,5944,0.053373,1,1.0



Validation interactions
------------------------------------------------------------------------------------------
Shape: (4760953, 6)
Memory usage: 113.51 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2020-02-11,45,83608,0.040661,2,1.0
1,2020-02-11,45,88947,0.033898,2,1.0
2,2020-02-11,45,73556,0.013542,2,1.0
3,2020-02-11,309,44193,0.013542,2,1.0
4,2020-02-11,309,87849,0.010153,2,1.0



Test interactions
------------------------------------------------------------------------------------------
Shape: (4792094, 6)
Memory usage: 114.25 MB


...

,article_id,article_idx
0,0108775015,0
1,0108775044,1
2,0108775051,2
3,0110065001,3
4,0110065002,4



Articles processed
------------------------------------------------------------------------------------------
Shape: (105542, 26)
Memory usage: 111.79 MB


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,article_idx
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,1
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,2
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",3
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",4



Sample submission processed
------------------------------------------------------------------------------------------
Shape: (1371980, 3)
Memory usage: 388.60 MB


,customer_id,prediction,customer_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0610776002 07...,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0610776002 07...,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0706016001 0706016002 0372860001 0610776002 07...,2
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0610776002 07...,3
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0610776002 07...,4


In [11]:
assert train_interactions["t_dat"].max() < val_interactions["t_dat"].min(), "Train/validation split overlap detected."
assert val_interactions["t_dat"].max() < test_interactions["t_dat"].min(), "Validation/test split overlap detected."

dataset_audit = {
    "train_rows": int(len(train_interactions)),
    "val_rows": int(len(val_interactions)),
    "test_rows": int(len(test_interactions)),
    "train_start": str(train_interactions["t_dat"].min().date()),
    "train_end": str(train_interactions["t_dat"].max().date()),
    "val_start": str(val_interactions["t_dat"].min().date()),
    "val_end": str(val_interactions["t_dat"].max().date()),
    "test_start": str(test_interactions["t_dat"].min().date()),
    "test_end": str(test_interactions["t_dat"].max().date()),
    "train_unique_users": int(train_interactions["customer_idx"].nunique()),
    "val_unique_users": int(val_interactions["customer_idx"].nunique()),
    "test_unique_users": int(test_interactions["customer_idx"].nunique()),
    "train_unique_items": int(train_interactions["article_idx"].nunique()),
    "val_unique_items": int(val_interactions["article_idx"].nunique()),
    "test_unique_items": int(test_interactions["article_idx"].nunique()),
}

print(json.dumps(dataset_audit, indent=2))
save_json(dataset_audit, BASELINE_ARTIFACTS_DIR / "popularity_dataset_audit.json")


{
  "train_rows": 22235277,
  "val_rows": 4760953,
  "test_rows": 4792094,
  "train_start": "2018-09-20",
  "train_end": "2020-02-10",
  "val_start": "2020-02-11",
  "val_end": "2020-06-08",
  "test_start": "2020-06-09",
  "test_end": "2020-09-22",
  "train_unique_users": 1150557,
  "val_unique_users": 574129,
  "test_unique_users": 578785,
  "train_unique_items": 83275,
  "val_unique_items": 43449,
  "test_unique_items": 44902
}
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/popularity_dataset_audit.json


## 5. Build popularity rankings from train only


In [12]:
def build_global_popularity(train_df: pd.DataFrame) -> pd.DataFrame:
    scores = (
        train_df
        .groupby("article_idx", as_index=False)
        .agg(
            popularity_score=("weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
            avg_train_price=("price", "mean"),
        )
    )

    scores["popularity_score"] = scores["popularity_score"].astype("float32")
    scores["purchase_count"] = scores["purchase_count"].astype("int32")
    scores["unique_customers"] = scores["unique_customers"].astype("int32")
    scores["avg_train_price"] = scores["avg_train_price"].astype("float32")

    scores = scores.sort_values(
        ["popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    scores["rank"] = np.arange(1, len(scores) + 1, dtype=np.int32)

    return scores


global_popularity = build_global_popularity(train_interactions)
global_ranked_items = global_popularity["article_idx"].to_numpy(dtype=np.int32)

print_df_info(global_popularity, "Global popularity from train")



Global popularity from train
------------------------------------------------------------------------------------------
Shape: (83275, 7)
Memory usage: 2.54 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank
0,53892,35373.0,35373,23701,2020-02-10,0.032287,1
1,53893,26288.0,26288,19317,2020-02-10,0.032531,2
2,1713,21688.0,21688,17997,2020-02-10,0.012916,3
3,2236,19882.0,19882,14449,2020-02-10,0.030726,4
4,14240,19642.0,19642,14369,2020-02-10,0.026013,5


In [13]:
def build_recent_popularity(
    train_df: pd.DataFrame,
    window_days: int,
    fallback_scores: pd.DataFrame,
) -> pd.DataFrame:
    max_date = train_df["t_dat"].max()
    cutoff_date = max_date - pd.Timedelta(days=window_days)
    recent_df = train_df[train_df["t_dat"] >= cutoff_date]

    scores = (
        recent_df
        .groupby("article_idx", as_index=False)
        .agg(
            popularity_score=("weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
            avg_train_price=("price", "mean"),
        )
    )

    scores["popularity_score"] = scores["popularity_score"].astype("float32")
    scores["purchase_count"] = scores["purchase_count"].astype("int32")
    scores["unique_customers"] = scores["unique_customers"].astype("int32")
    scores["avg_train_price"] = scores["avg_train_price"].astype("float32")

    scores = scores.sort_values(
        ["popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    existing_items = set(scores["article_idx"].to_numpy(dtype=np.int32))
    fallback_extra = fallback_scores[~fallback_scores["article_idx"].isin(existing_items)].copy()

    fallback_extra["popularity_score"] = 0.0
    fallback_extra["purchase_count"] = 0
    fallback_extra["unique_customers"] = 0

    combined = pd.concat(
        [
            scores,
            fallback_extra[
                [
                    "article_idx",
                    "popularity_score",
                    "purchase_count",
                    "unique_customers",
                    "last_train_purchase_date",
                    "avg_train_price",
                ]
            ],
        ],
        ignore_index=True,
    )

    combined["rank"] = np.arange(1, len(combined) + 1, dtype=np.int32)
    combined["window_days"] = window_days

    return combined


recent_popularity_by_window = {}

for window_days in RECENT_WINDOWS_DAYS:
    model_name = f"recent_{window_days}d_popularity"
    recent_popularity_by_window[model_name] = build_recent_popularity(
        train_df=train_interactions,
        window_days=window_days,
        fallback_scores=global_popularity,
    )
    print_df_info(recent_popularity_by_window[model_name], model_name)



recent_30d_popularity
------------------------------------------------------------------------------------------
Shape: (83275, 8)
Memory usage: 4.13 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,window_days
0,53892,2458.0,2458,2018,2020-02-10,0.032487,1,30
1,58491,2140.0,2140,1918,2020-02-10,0.032727,2,30
2,95395,2104.0,2104,1507,2020-02-10,0.024559,3,30
3,79861,2015.0,2015,1536,2020-02-10,0.029427,4,30
4,79863,1863.0,1863,1416,2020-02-10,0.016333,5,30



recent_90d_popularity
------------------------------------------------------------------------------------------
Shape: (83275, 8)
Memory usage: 4.13 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,window_days
0,53892,10015.0,10015,7755,2020-02-10,0.031087,1,90
1,58491,5411.0,5411,4768,2020-02-10,0.032285,2,90
2,53893,5209.0,5209,4307,2020-02-10,0.031360,3,90
3,73,4863.0,4863,2929,2020-02-10,0.011736,4,90
4,14253,4719.0,4719,3933,2020-02-10,0.029468,5,90


In [14]:
def build_recency_weighted_popularity(
    train_df: pd.DataFrame,
    half_life_days: float,
    fallback_scores: pd.DataFrame,
) -> pd.DataFrame:
    max_date = train_df["t_dat"].max()

    temp = train_df[["t_dat", "customer_idx", "article_idx", "price"]].copy()
    age_days = (max_date - temp["t_dat"]).dt.days.astype("float32")
    temp["recency_weight"] = np.power(0.5, age_days / half_life_days).astype("float32")

    scores = (
        temp
        .groupby("article_idx", as_index=False)
        .agg(
            popularity_score=("recency_weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
            avg_train_price=("price", "mean"),
        )
    )

    scores["popularity_score"] = scores["popularity_score"].astype("float32")
    scores["purchase_count"] = scores["purchase_count"].astype("int32")
    scores["unique_customers"] = scores["unique_customers"].astype("int32")
    scores["avg_train_price"] = scores["avg_train_price"].astype("float32")

    scores = scores.sort_values(
        ["popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    existing_items = set(scores["article_idx"].to_numpy(dtype=np.int32))
    fallback_extra = fallback_scores[~fallback_scores["article_idx"].isin(existing_items)].copy()

    fallback_extra["popularity_score"] = 0.0
    fallback_extra["purchase_count"] = 0
    fallback_extra["unique_customers"] = 0

    combined = pd.concat(
        [
            scores,
            fallback_extra[
                [
                    "article_idx",
                    "popularity_score",
                    "purchase_count",
                    "unique_customers",
                    "last_train_purchase_date",
                    "avg_train_price",
                ]
            ],
        ],
        ignore_index=True,
    )

    combined["rank"] = np.arange(1, len(combined) + 1, dtype=np.int32)
    combined["half_life_days"] = half_life_days

    return combined


recency_weighted_popularity = build_recency_weighted_popularity(
    train_df=train_interactions,
    half_life_days=HALF_LIFE_DAYS,
    fallback_scores=global_popularity,
)

recency_weighted_ranked_items = recency_weighted_popularity["article_idx"].to_numpy(dtype=np.int32)

print_df_info(recency_weighted_popularity, "Recency-weighted popularity")



Recency-weighted popularity
------------------------------------------------------------------------------------------
Shape: (83275, 8)
Memory usage: 3.81 MB


/tmp/ipykernel_16/1176743912.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,half_life_days
0,53892,4019.874756,35373,23701,2020-02-10,0.032287,1,30.0
1,58491,2614.516602,11755,9948,2020-02-10,0.032430,2,30.0
2,53893,2354.024658,26288,19317,2020-02-10,0.032531,3,30.0
3,9916,2091.709961,10352,8796,2020-02-10,0.015917,4,30.0
4,3711,1998.038330,17655,13520,2020-02-10,0.016168,5,30.0


## 5.1 Category / garment-group popularity baseline

This baseline recommends items from each user's most frequently purchased garment group, with global popularity as fallback.

In [15]:
def build_category_popularity(
    train_df: pd.DataFrame,
    articles_df: pd.DataFrame,
    category_col: str,
    fallback_scores: pd.DataFrame,
) -> Tuple[Dict[str, np.ndarray], Dict[int, str], pd.DataFrame]:
    """
    Build category-level popularity rankings and each user's most purchased category.
    Example category_col: garment_group_name.
    """
    if category_col not in articles_df.columns:
        raise ValueError(f"{category_col} not found in articles dataframe.")

    article_categories = articles_df[["article_idx", category_col]].copy()
    article_categories[category_col] = article_categories[category_col].fillna("Unknown")

    train_with_category = train_df.merge(
        article_categories,
        on="article_idx",
        how="left",
        validate="many_to_one",
    )

    train_with_category[category_col] = train_with_category[category_col].fillna("Unknown")

    category_scores = (
        train_with_category
        .groupby([category_col, "article_idx"], as_index=False)
        .agg(
            popularity_score=("weight", "sum"),
            purchase_count=("article_idx", "size"),
            unique_customers=("customer_idx", "nunique"),
            last_train_purchase_date=("t_dat", "max"),
        )
        .sort_values(
            [category_col, "popularity_score", "unique_customers", "last_train_purchase_date", "article_idx"],
            ascending=[True, False, False, False, True],
        )
        .reset_index(drop=True)
    )

    fallback_ranked_items = fallback_scores["article_idx"].to_numpy(dtype=np.int32)

    category_rankings = {}

    for category_value, group in category_scores.groupby(category_col, sort=False):
        category_items = group["article_idx"].to_numpy(dtype=np.int32)
        category_item_set = set(category_items)

        fallback_extra = [
            int(item)
            for item in fallback_ranked_items
            if int(item) not in category_item_set
        ]

        full_ranking = np.array(
            list(category_items) + fallback_extra,
            dtype=np.int32,
        )

        category_rankings[str(category_value)] = full_ranking

    user_category_counts = (
        train_with_category
        .groupby(["customer_idx", category_col], as_index=False)
        .size()
        .sort_values(["customer_idx", "size", category_col], ascending=[True, False, True])
    )

    user_primary_category = (
        user_category_counts
        .drop_duplicates("customer_idx")
        .set_index("customer_idx")[category_col]
        .astype(str)
        .to_dict()
    )

    print(f"Built {len(category_rankings):,} category rankings using {category_col}.")
    print(f"Built primary category for {len(user_primary_category):,} users.")

    return category_rankings, user_primary_category, category_scores


garment_group_rankings, user_primary_garment_group, garment_group_popularity_scores = build_category_popularity(
    train_df=train_interactions,
    articles_df=articles,
    category_col="garment_group_name",
    fallback_scores=global_popularity,
)

print_df_info(garment_group_popularity_scores, "Garment-group popularity scores")

Built 21 category rankings using garment_group_name.
Built primary category for 1,150,557 users.

Garment-group popularity scores
------------------------------------------------------------------------------------------
Shape: (83275, 6)
Memory usage: 7.30 MB


,garment_group_name,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date
0,Accessories,42557,15306.0,15306,13333,2020-02-10
1,Accessories,121,6683.0,6683,5966,2020-02-10
2,Accessories,40411,5786.0,5786,5325,2020-02-10
3,Accessories,13121,4820.0,4820,4589,2020-02-10
4,Accessories,7725,4744.0,4744,4423,2020-02-10


In [16]:
def recommend_from_category_ranking(
    user_id: int,
    user_primary_category: Dict[int, str],
    category_rankings: Dict[str, np.ndarray],
    fallback_ranked_items: np.ndarray,
    seen_items: Optional[set],
    k: int,
    n_candidates: int = 5000,
) -> List[int]:
    category_value = user_primary_category.get(user_id)

    if category_value in category_rankings:
        ranked_items = category_rankings[category_value]
    else:
        ranked_items = fallback_ranked_items

    return recommend_from_ranked_items(
        ranked_items=ranked_items,
        seen_items=seen_items,
        k=k,
        n_candidates=n_candidates,
    )


def evaluate_category_popularity(
    user_primary_category: Dict[int, str],
    category_rankings: Dict[str, np.ndarray],
    fallback_ranked_items: np.ndarray,
    train_seen_by_user: Dict[int, set],
    eval_actual_by_user: Dict[int, set],
    k_values: List[int],
    model_name: str,
    eval_name: str,
    warm_start_only: bool = False,
    total_catalog_items: Optional[int] = None,
) -> Tuple[pd.DataFrame, Dict[int, List[int]]]:
    started = time.time()

    if warm_start_only:
        user_ids = [u for u in eval_actual_by_user.keys() if u in train_seen_by_user]
    else:
        user_ids = list(eval_actual_by_user.keys())

    if not user_ids:
        raise ValueError(f"No users found for {model_name}, {eval_name}, warm_start={warm_start_only}")

    max_k = max(k_values)

    sums = {
        k: {
            "precision": 0.0,
            "recall": 0.0,
            "map": 0.0,
            "ndcg": 0.0,
            "mrr": 0.0,
            "hitrate": 0.0,
        }
        for k in k_values
    }

    unique_recommended_items_by_k = {k: set() for k in k_values}
    sample_recommendations_by_user = {}

    for idx, user_id in enumerate(user_ids):
        actual = eval_actual_by_user[user_id]
        seen = train_seen_by_user.get(user_id)

        recommended = recommend_from_category_ranking(
            user_id=int(user_id),
            user_primary_category=user_primary_category,
            category_rankings=category_rankings,
            fallback_ranked_items=fallback_ranked_items,
            seen_items=seen,
            k=max_k,
            n_candidates=N_POPULAR_CANDIDATES,
        )

        if idx < 10:
            sample_recommendations_by_user[int(user_id)] = recommended

        for k in k_values:
            top_k_recommended = recommended[:k]
            unique_recommended_items_by_k[k].update(top_k_recommended)

            sums[k]["precision"] += precisionk(recommended, actual, k)
            sums[k]["recall"] += recallk(recommended, actual, k)
            sums[k]["map"] += apk(recommended, actual, k)
            sums[k]["ndcg"] += ndcgk(recommended, actual, k)
            sums[k]["mrr"] += mrrk(recommended, actual, k)
            sums[k]["hitrate"] += hitratek(recommended, actual, k)

    rows = []
    n_users = len(user_ids)
    elapsed_seconds = time.time() - started

    for k in k_values:
        row = {
            "model_name": model_name,
            "eval_split": eval_name,
            "warm_start_only": warm_start_only,
            "k": k,
            "n_eval_users": n_users,
            "precision_at_k": sums[k]["precision"] / n_users,
            "recall_at_k": sums[k]["recall"] / n_users,
            "map_at_k": sums[k]["map"] / n_users,
            "ndcg_at_k": sums[k]["ndcg"] / n_users,
            "mrr_at_k": sums[k]["mrr"] / n_users,
            "hitrate_at_k": sums[k]["hitrate"] / n_users,
            "unique_recommended_items": len(unique_recommended_items_by_k[k]),
            "elapsed_seconds": round(elapsed_seconds, 2),
        }

        if total_catalog_items is not None and total_catalog_items > 0:
            row["catalog_coverage"] = len(unique_recommended_items_by_k[k]) / total_catalog_items
        else:
            row["catalog_coverage"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows), sample_recommendations_by_user


In [17]:
popularity_rankings = {
    "global_popularity": global_popularity["article_idx"].to_numpy(dtype=np.int32),
    "recency_weighted_popularity": recency_weighted_popularity["article_idx"].to_numpy(dtype=np.int32),
}

for model_name, scores_df in recent_popularity_by_window.items():
    popularity_rankings[model_name] = scores_df["article_idx"].to_numpy(dtype=np.int32)

print("Popularity models:")
for name, ranking in popularity_rankings.items():
    print(f"{name}: {len(ranking):,} ranked items")


Popularity models:
global_popularity: 83,275 ranked items
recency_weighted_popularity: 83,275 ranked items
recent_30d_popularity: 83,275 ranked items
recent_90d_popularity: 83,275 ranked items


## 6. Save popularity score tables


In [18]:
article_idx_to_id = article_id_map[["article_idx", "article_id"]].copy()


def attach_article_ids(scores_df: pd.DataFrame) -> pd.DataFrame:
    return scores_df.merge(article_idx_to_id, on="article_idx", how="left", validate="many_to_one")


global_popularity_out = attach_article_ids(global_popularity)
recency_weighted_popularity_out = attach_article_ids(recency_weighted_popularity)

save_parquet(global_popularity_out, BASELINE_ARTIFACTS_DIR / "global_popularity_scores.parquet")
save_parquet(recency_weighted_popularity_out, BASELINE_ARTIFACTS_DIR / "recency_weighted_popularity_scores.parquet")

for model_name, scores_df in recent_popularity_by_window.items():
    save_parquet(
        attach_article_ids(scores_df),
        BASELINE_ARTIFACTS_DIR / f"{model_name}_scores.parquet",
    )


garment_group_popularity_scores_out = attach_article_ids(garment_group_popularity_scores)
save_parquet(
    garment_group_popularity_scores_out,
    BASELINE_ARTIFACTS_DIR / "garment_group_popularity_scores.parquet",
)

user_primary_garment_group_df = pd.DataFrame(
    [
        {"customer_idx": int(user_id), "primary_garment_group_name": category}
        for user_id, category in user_primary_garment_group.items()
    ]
)
save_parquet(
    user_primary_garment_group_df,
    BASELINE_ARTIFACTS_DIR / "user_primary_garment_group.parquet",
)


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/global_popularity_scores.parquet | 2.27 MB
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/recency_weighted_popularity_scores.parquet | 2.87 MB
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/recent_30d_popularity_scores.parquet | 2.11 MB
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/recent_90d_popularity_scores.parquet | 2.13 MB
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/garment_group_popularity_scores.parquet | 1.42 MB
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/user_primary_garment_group.parquet | 5.55 MB


## 7. Build user target sets and train seen sets


In [19]:
print("Building user-item sets. This can take a few minutes on the full H&M data.")
started = time.time()

train_seen_by_user = build_user_item_sets(train_interactions)
val_actual_by_user = build_user_item_sets(val_interactions)
test_actual_by_user = build_user_item_sets(test_interactions)

elapsed = time.time() - started
print(f"Built dictionaries in {elapsed:.2f} seconds.")
print("Train users with seen items:", len(train_seen_by_user))
print("Validation users with target items:", len(val_actual_by_user))
print("Test users with target items:", len(test_actual_by_user))


Building user-item sets. This can take a few minutes on the full H&M data.
Built dictionaries in 55.93 seconds.
Train users with seen items: 1150557
Validation users with target items: 574129
Test users with target items: 578785


In [20]:
target_set_audit = {
    "train_seen_users": len(train_seen_by_user),
    "val_target_users": len(val_actual_by_user),
    "test_target_users": len(test_actual_by_user),
    "val_warm_users": sum(1 for u in val_actual_by_user if u in train_seen_by_user),
    "test_warm_users": sum(1 for u in test_actual_by_user if u in train_seen_by_user),
    "val_cold_users": sum(1 for u in val_actual_by_user if u not in train_seen_by_user),
    "test_cold_users": sum(1 for u in test_actual_by_user if u not in train_seen_by_user),
}

print(json.dumps(target_set_audit, indent=2))
save_json(target_set_audit, BASELINE_ARTIFACTS_DIR / "target_set_audit.json")


{
  "train_seen_users": 1150557,
  "val_target_users": 574129,
  "test_target_users": 578785,
  "val_warm_users": 450725,
  "test_warm_users": 456034,
  "val_cold_users": 123404,
  "test_cold_users": 122751
}
Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/target_set_audit.json


## 8. Evaluate popularity baselines


In [21]:
all_metric_frames = []
all_sample_recommendations = {}

total_catalog_items = int(train_interactions["article_idx"].nunique())

for model_name, ranked_items in popularity_rankings.items():
    print()
    print(f"Evaluating {model_name}")

    for eval_name, actual_by_user in [
        ("validation", val_actual_by_user),
        ("test", test_actual_by_user),
    ]:
        for warm_start_only in [False, True]:
            metrics_df, sample_recs = evaluate_fixed_ranking(
                ranked_items=ranked_items,
                train_seen_by_user=train_seen_by_user,
                eval_actual_by_user=actual_by_user,
                k_values=TOP_K_VALUES,
                model_name=model_name,
                eval_name=eval_name,
                warm_start_only=warm_start_only,
                total_catalog_items=total_catalog_items,
            )

            all_metric_frames.append(metrics_df)
            sample_key = f"{model_name}_{eval_name}_{'warm' if warm_start_only else 'full'}"
            all_sample_recommendations[sample_key] = sample_recs
            display(metrics_df)

print()
print("Evaluating garment_group_popularity")

for eval_name, actual_by_user in [
    ("validation", val_actual_by_user),
    ("test", test_actual_by_user),
]:
    for warm_start_only in [False, True]:
        metrics_df, sample_recs = evaluate_category_popularity(
            user_primary_category=user_primary_garment_group,
            category_rankings=garment_group_rankings,
            fallback_ranked_items=global_ranked_items,
            train_seen_by_user=train_seen_by_user,
            eval_actual_by_user=actual_by_user,
            k_values=TOP_K_VALUES,
            model_name="garment_group_popularity",
            eval_name=eval_name,
            warm_start_only=warm_start_only,
            total_catalog_items=total_catalog_items,
        )

        all_metric_frames.append(metrics_df)
        sample_key = f"garment_group_popularity_{eval_name}_{'warm' if warm_start_only else 'full'}"
        all_sample_recommendations[sample_key] = sample_recs
        display(metrics_df)

metrics = pd.concat(all_metric_frames, ignore_index=True)

metric_cols = [
    "precision_at_k",
    "recall_at_k",
    "map_at_k",
    "ndcg_at_k",
    "mrr_at_k",
    "hitrate_at_k",
    "catalog_coverage",
]
metrics[metric_cols] = metrics[metric_cols].round(6)

metrics_sorted = metrics.sort_values(
    ["eval_split", "warm_start_only", "k", "recall_at_k", "ndcg_at_k", "mrr_at_k"],
    ascending=[True, True, True, False, False, False],
).reset_index(drop=True)

display(metrics_sorted)



Evaluating global_popularity


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,validation,False,12,574129,0.004256,0.009268,0.003919,0.008289,0.018154,0.047505,23,17.32,0.000276
1,global_popularity,validation,False,20,574129,0.004292,0.015212,0.004157,0.010350,0.019971,0.076734,32,17.32,0.000384


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,validation,True,12,450725,0.004478,0.008793,0.003647,0.008168,0.018989,0.050017,23,14.21,0.000276
1,global_popularity,validation,True,20,450725,0.004488,0.014219,0.003822,0.010070,0.020863,0.080044,32,14.21,0.000384


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,test,False,12,578785,0.003160,0.006673,0.002554,0.005781,0.012786,0.036063,23,17.35,0.000276
1,global_popularity,test,False,20,578785,0.003126,0.010669,0.002715,0.007196,0.014063,0.056672,32,17.35,0.000384


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,test,True,12,456034,0.003300,0.006388,0.002425,0.005722,0.013201,0.037611,23,14.29,0.000276
1,global_popularity,test,True,20,456034,0.003274,0.010242,0.002559,0.007098,0.014550,0.059329,32,14.29,0.000384


...

,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,garment_group_popularity,validation,False,12,574129,0.003095,0.006662,0.003011,0.006184,0.013608,0.033616,395,18.55,0.004743
1,garment_group_popularity,validation,False,20,574129,0.002633,0.009668,0.003090,0.007081,0.014408,0.046462,590,18.55,0.007085


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,garment_group_popularity,validation,True,12,450725,0.002999,0.005473,0.002491,0.005487,0.013198,0.032326,395,15.04,0.004743
1,garment_group_popularity,validation,True,20,450725,0.002374,0.007158,0.002463,0.005906,0.013777,0.041484,590,15.04,0.007085


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,garment_group_popularity,test,False,12,578785,0.002238,0.004728,0.001940,0.004232,0.009380,0.024995,395,18.66,0.004743
1,garment_group_popularity,test,False,20,578785,0.001816,0.006511,0.001981,0.004753,0.009878,0.032973,589,18.66,0.007073


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,garment_group_popularity,test,True,12,456034,0.002129,0.003920,0.001645,0.003757,0.008878,0.023564,395,15.29,0.004743
1,garment_group_popularity,test,True,20,456034,0.001611,0.004964,0.001627,0.003997,0.009238,0.029250,589,15.29,0.007073


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,test,False,12,578785,0.003160,0.006673,0.002554,0.005781,0.012786,0.036063,23,17.35,0.000276
1,recency_weighted_popularity,test,False,12,578785,0.002524,0.005208,0.002289,0.004885,0.011199,0.027978,21,17.38,0.000252
2,recent_90d_popularity,test,False,12,578785,0.002432,0.005095,0.002207,0.004718,0.010868,0.026830,21,17.52,0.000252
3,garment_group_popularity,test,False,12,578785,0.002238,0.004728,0.001940,0.004232,0.009380,0.024995,395,18.66,0.004743
4,recent_30d_popularity,test,False,12,578785,0.001920,0.003864,0.001864,0.003847,0.009430,0.021006,19,17.01,0.000228
5,global_popularity,test,False,20,578785,0.003126,0.010669,0.002715,0.007196,0.014063,0.056672,32,17.35,0.000384
6,recent_90d_popularity,test,False,20,578785,0.002177,0.007565,0.002275,0.005530,0.011746,0.040150,30,17.52,0.000360
7,recency_weighted_popularity,test,False,20,578785,0.002191,0.007498,0.002334,0.005608,0.012009,0.040530,32,17.38,0.000384
8,recent_30d_popularity,test,False,20,578785,0.002126,0.007095,0.001990,0.005004,0.010591,0.039232,28,17.01,0.000336
9,garment_group_popularity,test,False,20,578785,0.001816,0.006511,0.001981,0.004753,0.009878,0.032973,589,18.66,0.007073


In [22]:
save_csv(metrics, BASELINE_ARTIFACTS_DIR / "popularity_baseline_metrics.csv")

primary_leaderboard = (
    metrics[
        (metrics["eval_split"] == "validation")
        & (metrics["warm_start_only"] == True)
        & (metrics["k"] == PRIMARY_K)
    ]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k"], ascending=False)
    .reset_index(drop=True)
)

display(primary_leaderboard)

best_model_name = str(primary_leaderboard.loc[0, "model_name"])

best_model_info = {
    "best_model_name": best_model_name,
    "selection_split": "validation",
    "selection_mode": "warm_start_only",
    "selection_k": PRIMARY_K,
    "selection_metric": "recall_at_k",
    "best_validation_metrics": primary_leaderboard.iloc[0].to_dict(),
}

save_json(best_model_info, BASELINE_ARTIFACTS_DIR / "best_popularity_model_info.json")

print("Best popularity model:", best_model_name)


def recommend_from_selected_popularity_model(
    user_id: int,
    model_name: str,
    train_seen_by_user: Dict[int, set],
    k: int,
) -> List[int]:
    """Recommend from the selected baseline, including user-dependent garment-group popularity."""
    seen = train_seen_by_user.get(int(user_id))

    if model_name == "garment_group_popularity":
        return recommend_from_category_ranking(
            user_id=int(user_id),
            user_primary_category=user_primary_garment_group,
            category_rankings=garment_group_rankings,
            fallback_ranked_items=global_ranked_items,
            seen_items=seen,
            k=k,
            n_candidates=N_POPULAR_CANDIDATES,
        )

    if model_name not in popularity_rankings:
        raise ValueError(f"Unknown popularity model: {model_name}")

    return recommend_from_ranked_items(
        ranked_items=popularity_rankings[model_name],
        seen_items=seen,
        k=k,
        n_candidates=N_POPULAR_CANDIDATES,
    )


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/popularity_baseline_metrics.csv | 0.00 MB


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
0,global_popularity,validation,True,20,450725,0.004488,0.014219,0.003822,0.010070,0.020863,0.080044,32,14.21,0.000384
1,recent_30d_popularity,validation,True,20,450725,0.004149,0.012871,0.003816,0.009641,0.021224,0.073193,29,13.85,0.000348
2,recency_weighted_popularity,validation,True,20,450725,0.003848,0.012608,0.004140,0.009861,0.022009,0.069262,32,14.12,0.000384
3,recent_90d_popularity,validation,True,20,450725,0.003638,0.012396,0.004066,0.009603,0.021494,0.065725,30,14.24,0.000360
4,garment_group_popularity,validation,True,20,450725,0.002374,0.007158,0.002463,0.005906,0.013777,0.041484,590,15.04,0.007085


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/best_popularity_model_info.json
Best popularity model: global_popularity


## 9. Inspect top recommended articles


In [23]:
preview_user_ids = (
    sample_submission["customer_idx"]
    .dropna()
    .astype("int32")
    .head(1)
    .to_numpy()
)

if len(preview_user_ids) > 0:
    preview_user_id = int(preview_user_ids[0])
    best_preview_items = recommend_from_selected_popularity_model(
        user_id=preview_user_id,
        model_name=best_model_name,
        train_seen_by_user=train_seen_by_user,
        k=50,
    )
    print(f"Showing top 50 recommendations for preview user {preview_user_id} using {best_model_name}.")
else:
    preview_user_id = None
    best_preview_items = global_ranked_items[:50].astype(int).tolist()
    print("No processed sample-submission users found. Showing global popularity top 50.")

top_items_df = pd.DataFrame(
    {
        "customer_idx": preview_user_id,
        "article_idx": best_preview_items,
        "rank": np.arange(1, len(best_preview_items) + 1, dtype=np.int32),
        "model_name": best_model_name,
    }
)

top_items_df = top_items_df.merge(
    article_id_map[["article_idx", "article_id"]],
    on="article_idx",
    how="left",
    validate="many_to_one",
)

preview_cols = [
    "article_idx",
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "department_name",
    "index_name",
    "section_name",
    "garment_group_name",
]

article_cols = [col for col in preview_cols if col in articles.columns]

top_items_df = top_items_df.merge(
    articles[article_cols],
    on=["article_idx", "article_id"],
    how="left",
    validate="many_to_one",
)

display(top_items_df.head(50))
save_parquet(top_items_df, BASELINE_ARTIFACTS_DIR / "top_50_popularity_recommendations.parquet")


Showing top 50 recommendations for preview user 0 using global_popularity.


,customer_idx,article_idx,rank,model_name,article_id,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,department_name,index_name,section_name,garment_group_name
0,0,53892,1,global_popularity,0706016001,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Solid,Black,Trousers,Divided,Divided Collection,Trousers
1,0,53893,2,global_popularity,0706016002,Jade HW Skinny Denim TRS,Trousers,Garment Lower body,Solid,Light Blue,Trousers,Divided,Divided Collection,Trousers
2,0,1713,3,global_popularity,0372860001,7p Basic Shaftless,Socks,Socks & Tights,Solid,Black,Shopbasket Socks,Lingeries/Tights,"Womens Nightwear, Socks & Tigh",Socks and Tights
3,0,2236,4,global_popularity,0399223001,Curvy Jeggings HW Ankle,Trousers,Garment Lower body,Solid,Black,Denim Trousers,Divided,Ladies Denim,Trousers Denim
4,0,14240,5,global_popularity,0562245001,Luna skinny RW,Trousers,Garment Lower body,Solid,Black,Trouser,Ladieswear,Womens Everyday Collection,Trousers
5,0,67,6,global_popularity,0156231001,Box 4p Tights,Underwear Tights,Socks & Tights,Solid,Black,Tights basic,Lingeries/Tights,"Womens Nightwear, Socks & Tigh",Socks and Tights
6,0,3711,7,global_popularity,0464297007,Greta Thong Mynta Low 3p,Underwear bottom,Underwear,Placement print,Black,Casual Lingerie,Lingeries/Tights,Womens Lingerie,"Under-, Nightwear"
7,0,24837,8,global_popularity,0610776002,Tilly (1),T-shirt,Garment Upper body,Solid,Black,Jersey Basic,Ladieswear,Womens Everyday Basics,Jersey Basic
8,0,42626,9,global_popularity,0673677002,Henry polo. (1),Sweater,Garment Upper body,Solid,Black,Knitwear,Ladieswear,Womens Tailoring,Knitwear
9,0,2252,10,global_popularity,0399256001,Skinny Ankle R.W Brooklyn,Trousers,Garment Lower body,Solid,Black,Everyday Waredrobe Denim,Ladieswear,Ladies Denim,Trousers Denim


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/top_50_popularity_recommendations.parquet | 0.01 MB


## 10. Create demo recommendations


In [24]:
def build_recommendations_for_users(
    user_ids: Iterable[int],
    model_name: str,
    train_seen_by_user: Dict[int, set],
    k: int,
) -> pd.DataFrame:
    rows = []

    for user_id in user_ids:
        user_id_int = int(user_id)
        recs = recommend_from_selected_popularity_model(
            user_id=user_id_int,
            model_name=model_name,
            train_seen_by_user=train_seen_by_user,
            k=k,
        )

        for rank, item_idx in enumerate(recs, start=1):
            rows.append(
                {
                    "customer_idx": user_id_int,
                    "article_idx": int(item_idx),
                    "rank": rank,
                    "model_name": model_name,
                }
            )

    return pd.DataFrame(rows)


demo_user_ids = (
    sample_submission["customer_idx"]
    .dropna()
    .astype("int32")
    .head(N_DEMO_USERS)
    .to_numpy()
)

demo_recommendations = build_recommendations_for_users(
    user_ids=demo_user_ids,
    model_name=best_model_name,
    train_seen_by_user=train_seen_by_user,
    k=PRIMARY_K,
)

demo_recommendations = demo_recommendations.merge(
    article_id_map[["article_idx", "article_id"]],
    on="article_idx",
    how="left",
    validate="many_to_one",
)

print_df_info(demo_recommendations, "Demo popularity recommendations")
save_parquet(demo_recommendations, BASELINE_ARTIFACTS_DIR / "demo_popularity_recommendations.parquet")



Demo popularity recommendations
------------------------------------------------------------------------------------------
Shape: (20000, 5)
Memory usage: 2.84 MB


,customer_idx,article_idx,rank,model_name,article_id
0,0,53892,1,global_popularity,0706016001
1,0,53893,2,global_popularity,0706016002
2,0,1713,3,global_popularity,0372860001
3,0,2236,4,global_popularity,0399223001
4,0,14240,5,global_popularity,0562245001


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/demo_popularity_recommendations.parquet | 0.01 MB


## 11. Optional Kaggle-style prediction file


In [25]:
def build_kaggle_style_predictions(
    users_df: pd.DataFrame,
    model_name: str,
    train_seen_by_user: Dict[int, set],
    k: int = 12,
) -> pd.DataFrame:
    article_idx_to_article_id = dict(
        zip(
            article_id_map["article_idx"].astype(int),
            article_id_map["article_id"].astype(str),
        )
    )

    rows = []

    for row in users_df[["customer_id", "customer_idx"]].itertuples(index=False):
        customer_id = row.customer_id
        customer_idx = int(row.customer_idx)

        recs = recommend_from_selected_popularity_model(
            user_id=customer_idx,
            model_name=model_name,
            train_seen_by_user=train_seen_by_user,
            k=k,
        )

        prediction = " ".join(article_idx_to_article_id[int(item)] for item in recs)

        rows.append(
            {
                "customer_id": customer_id,
                "prediction": prediction,
            }
        )

    return pd.DataFrame(rows)


if GENERATE_FULL_SAMPLE_SUBMISSION:
    started = time.time()

    full_prediction_df = build_kaggle_style_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]),
        model_name=best_model_name,
        train_seen_by_user=train_seen_by_user,
        k=12,
    )

    save_csv(full_prediction_df, BASELINE_ARTIFACTS_DIR / "popularity_baseline_submission_like.csv")
    print(f"Generated full sample-submission-style file in {time.time() - started:.2f} seconds.")

else:
    small_prediction_df = build_kaggle_style_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]).head(N_DEMO_USERS),
        model_name=best_model_name,
        train_seen_by_user=train_seen_by_user,
        k=12,
    )

    display(small_prediction_df.head())
    save_csv(small_prediction_df, BASELINE_ARTIFACTS_DIR / "popularity_baseline_submission_like_demo.csv")
    print("Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.")


,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0399223001 05...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0399223001 05...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0706016001 0706016002 0372860001 0399223001 05...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0399223001 05...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0399223001 05...


Saved: /kaggle/working/hm-recommender/artifacts/popularity_baseline/popularity_baseline_submission_like_demo.csv | 0.19 MB
Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.


## 12. Final summary


In [26]:
final_summary = {
    "best_model_name": best_model_name,
    "artifact_dir": str(BASELINE_ARTIFACTS_DIR),
    "included_fixes": [
        "MRR@K added",
        "catalog coverage calculated separately for each K",
        "garment-group popularity baseline added",
        "selected baseline helper supports user-dependent garment-group recommendations",
    ],
    "saved_files": sorted([p.name for p in BASELINE_ARTIFACTS_DIR.glob("*")]),
}

print(json.dumps(final_summary, indent=2))

print()
print("Popularity baseline completed successfully.")
print("Next step: 04-als-recommender.ipynb")


{
  "best_model_name": "global_popularity",
  "artifact_dir": "/kaggle/working/hm-recommender/artifacts/popularity_baseline",
  "included_fixes": [
    "MRR@K added",
    "catalog coverage calculated separately for each K",
    "garment-group popularity baseline added",
    "selected baseline helper supports user-dependent garment-group recommendations"
  ],
  "saved_files": [
    "best_popularity_model_info.json",
    "demo_popularity_recommendations.parquet",
    "garment_group_popularity_scores.parquet",
    "global_popularity_scores.parquet",
    "popularity_baseline_metrics.csv",
    "popularity_baseline_submission_like_demo.csv",
    "popularity_dataset_audit.json",
    "recency_weighted_popularity_scores.parquet",
    "recent_30d_popularity_scores.parquet",
    "recent_90d_popularity_scores.parquet",
    "target_set_audit.json",
    "top_50_popularity_recommendations.parquet",
    "user_primary_garment_group.parquet"
  ]
}

Popularity baseline completed successfully.
Next step: 